In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import shutil
import string
from sklearn.metrics import classification_report
import tensorflow as tf
from keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras import losses
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping,ModelCheckpoint
from tensorflow.keras.layers import Dense, Input, Dropout, Bidirectional, LSTM, Embedding, BatchNormalization,  Reshape, Conv2D, MaxPool2D, concatenate, Flatten, Activation
import torch
import numpy as np
from transformers import BertTokenizer, BertModel,RobertaTokenizer, RobertaModel
import ast

In [ ]:
# Verificar si la GPU está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
archivo_3 = '/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/DataProgramsandDescriptions-CatRetroalimentacion5000.xlsx'
train_full = pd.read_excel(archivo_3)

# AST spliter D. Gries form

In [ ]:
class WhileLoopFinder(ast.NodeVisitor):
    def __init__(self, source_code):
        self.source_code = source_code.splitlines()
        self.functions_with_while = []
        self.current_function = None

    def visit_FunctionDef(self, node):
        original_current_function = self.current_function
        self.current_function = {
            "function_name": node.name,
            "has_while_loop": False,
            "pre_while_code_lines": [],
            "while_loops": []
        }

        function_start_line = node.lineno - 1
        function_end_line = (node.end_lineno if hasattr(node, 'end_lineno') else len(self.source_code))
        function_lines = self.source_code[function_start_line:function_end_line]

        found_while = False
        for stmt in node.body:
            if isinstance(stmt, ast.While):
                self.current_function["has_while_loop"] = True
                found_while = True
                try:
                    while_condition = ast.unparse(stmt.test).strip()
                    print("OK condition")
                except AttributeError:
                    print("Error al obtener la condición del while")
                try:
                    while_body_code = ast.unparse(ast.Module(body=stmt.body, type_ignores=[])).strip()
                    print("Ok body")
                except AttributeError:
                    print("Error al obtener el código del cuerpo del while")

                self.current_function["while_loops"].append({
                    "condition": while_condition,
                    "body_code": while_body_code
                })
            elif not found_while:
                start_line = stmt.lineno - 1
                end_line = (stmt.end_lineno if hasattr(stmt, 'end_lineno') else stmt.lineno)
                self.current_function["pre_while_code_lines"].extend(self.source_code[start_line:end_line])
            self.generic_visit(stmt)

        if self.current_function["has_while_loop"]:
            last_while_end_line = None
            if self.current_function["while_loops"]:
                last_while_node = next((node for node in reversed(node.body) if isinstance(node, ast.While)), None)
                if last_while_node and hasattr(last_while_node, 'end_lineno'):
                    last_while_end_line = last_while_node.end_lineno
            post_while_code_lines = []
            if last_while_end_line:
                function_indent = len(self.source_code[node.lineno - 1]) - len(self.source_code[node.lineno - 1].lstrip())
                for line in self.source_code[last_while_end_line:function_end_line]:
                    if line.startswith(self.source_code[node.lineno - 1][:function_indent] + "    "):
                        post_while_code_lines.append(line[function_indent + 4:])
                    else:
                        post_while_code_lines.append(line[function_indent:])
            self.current_function["post_while_code_lines"] = "\n".join(post_while_code_lines).strip()
            self.current_function["pre_while_code_lines"] = "\n".join(self.current_function["pre_while_code_lines"]).strip()
            self.functions_with_while.append(self.current_function)

        self.current_function = original_current_function



In [ ]:
def DGries_states(solution):
  try:
    # Crear el AST
    tree = ast.parse(solution)
    # Recorrer el AST con nuestro visitante
    finder = WhileLoopFinder(solution)
    finder.visit(tree)
    for func_info in finder.functions_with_while:
      initial_state=func_info["pre_while_code_lines"] if func_info["pre_while_code_lines"] else False
      end_state=""
      transformation_state=""
      for j, wl in enumerate(func_info["while_loops"]):
          end_state=wl["condition"]
          transformation_state=wl["body_code"]

      if not initial_state:
        initial_state=transformation_state
      return initial_state,transformation_state,end_state
  except Exception as error:
    return "Exception:"+str(error),"Exception:"+str(error),"Exception:"+str(error)




# Encoder Description and Code

In [ ]:
class EncodeTextSource:
  def __init__(self):
    self.is_loadtokenizers=False



  def tokenize_and_generate_embeddings_descriptions(self,description):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input description and move tokens to GPU
    tokens = self.beart_tokenizer.encode_plus(description, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.beart_model(**tokens)
    # Extract embeddings for all tokens
    desc_embeddings = outputs.last_hidden_state.cpu().numpy()
    return desc_embeddings

  def tokenize_and_generate_embeddings_codes(self,code):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input code and move tokens to GPU
    tokens = self.beart_tokenizer.encode_plus(code, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.codebeart_model(**tokens)
    # Extract embeddings for all tokens
    code_embeddings = outputs.last_hidden_state.cpu().numpy()

    return code_embeddings

  def load_beart_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    model_name = "bert-base-uncased"
    self.beart_model = BertModel.from_pretrained(model_name)
    self.beart_tokenizer = BertTokenizer.from_pretrained(model_name)
    self.beart_model.to(device)

  def load_codebert_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    self.codebeart_tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
    self.codebeart_model = RobertaModel.from_pretrained("microsoft/codebert-base")
    self.codebeart_model.to(device)

  def start_tokenizer(self):
    self.load_beart_tokenizer()
    self.load_codebert_tokenizer()
    self.is_loadtokenizers=True




# All characteristics

In [ ]:
def categoricallabelAll(w):

  if w=="['Initial state']":
    return 0
  if w=="['Final state']":
    return 1
  if w=="['State transformation']":
    return 2
  if w=="['Initial state', 'Final state']":
    return 3
  if w=="['Initial state', 'State transformation']":
    return 4
  if w=="['Final state', 'State transformation']":
    return 5
  if w=="['Initial state', 'Final state', 'State transformation']":
    return 6
  return 7

category=np.array([
    'Initial state',
    'Final state',
    'State transformation',
    'Initial state, Final state',
    'Initial state, State transformation',
    'Final state, State transformation',
    'Initial state, Final state, State transformation'

])

# Load Dataset

In [ ]:
train_full.head()

,No.,Problema,Solución,Estado incial,Estado final,Transformación de estado,Etiqueta 1,Etiqueta 2,Realimentación
0,1,Write a Python function that returns the facto...,def factorial(n):\n result = 1\n i = 1\n...,"result = 1, i = 1",i <= n,"result *= i, i += 1",Correct,['Correct'],NaN
1,2,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,"total = 0, i = 1, n = 100",i <= n,"total += i, i += 1",Correct,['Correct'],NaN
2,3,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,"numbers = [], i = 0, n = 10",i <= n,"print(i), numbers.append(i), i += 1",Correct,['Correct'],NaN
3,4,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,"numbers = [], i = 10, n = 1",i >= n,"print(i), numbers.append(i), i -= 1",Correct,['Correct'],NaN
4,5,Write a Python function to check if a number i...,def is_prime(num):\n if num <= 1:\n ...,"i = 2, i = = 0:",i <= num//2,"if num % i == 0:, return False, i += 1",Correct,['Correct'],NaN


In [ ]:
train_full.drop(["No.","Realimentación","Estado incial","Estado final","Transformación de estado"],axis=1,inplace=True)
train_full

,Problema,Solución,Etiqueta 1,Etiqueta 2
0,Write a Python function that returns the facto...,def factorial(n):\n result = 1\n i = 1\n...,Correct,['Correct']
1,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,Correct,['Correct']
2,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,Correct,['Correct']
3,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,Correct,['Correct']
4,Write a Python function to check if a number i...,def is_prime(num):\n if num <= 1:\n ...,Correct,['Correct']
...,...,...,...,...
4995,Write a Python function that returns the sum o...,def sum_of_first_five_numbers():\n number =...,Incorrect,"['Initial state', 'Final state']"
4996,Write a Python function that returns the facto...,def factorial(n):\n result = 0\n i = 0\n...,Incorrect,"['Initial state', 'Final state']"
4997,Write a Python function that prints the number...,def print_and_store_numbers():\n numbers = ...,Incorrect,"['Initial state', 'Final state']"
4998,Write a Python function to print the numbers f...,def print_and_store_numbers():\n numbers = ...,Incorrect,"['Initial state', 'Final state']"


In [ ]:
train_full[["Estado incial","Transformación de estado","Estado final"]]=train_full['Solución'].apply(DGries_states).apply(pd.Series)

Streaming output truncated to the last 5000 lines.
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK c

In [ ]:
train_full = train_full[train_full['Etiqueta 1']!='Correct'].copy()

In [ ]:
y=train_full['Etiqueta 2'].apply(categoricallabelAll)
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6])

In [ ]:
train_full.dropna(inplace=True)

In [ ]:
np.unique(train_full['Etiqueta 2'])

array(["['Final state', 'State transformation']", "['Final state']",
       "['Initial state', 'Final state', 'State transformation']",
       "['Initial state', 'Final state']",
       "['Initial state', 'State transformation']", "['Initial state']",
       "['State transformation']"], dtype=object)

In [ ]:
encoder=EncodeTextSource()
encoder.start_tokenizer()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [ ]:
%%time
problem=train_full['Problema'].apply(encoder.tokenize_and_generate_embeddings_descriptions).to_numpy()
startstate = train_full['Estado incial'].apply(encoder.tokenize_and_generate_embeddings_codes).to_numpy()
finalstate = train_full['Estado final'].apply(encoder.tokenize_and_generate_embeddings_codes).to_numpy()
transstate = train_full['Transformación de estado'].apply(encoder.tokenize_and_generate_embeddings_codes).to_numpy()

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

CPU times: user 1min 51s, sys: 2.26 s, total: 1min 53s
Wall time: 1min 50s


In [ ]:
problem.shape,startstate.shape,finalstate.shape,transstate.shape

((3459,), (3459,), (3459,), (3459,))

In [ ]:
print(startstate.shape)
print(startstate[1262].shape)


(3459,)
(1, 9, 768)


In [ ]:
Xp=np.array([sentence[0].mean(axis=0) for sentence in problem])
Xs=np.array([sentence[0].mean(axis=0) for sentence in startstate])
Xf=np.array([sentence[0].mean(axis=0) for sentence in finalstate])
Xt=np.array([sentence[0].mean(axis=0) for sentence in transstate])
Xp.shape,Xs.shape,Xf.shape,Xt.shape

((3459, 768), (3459, 768), (3459, 768), (3459, 768))

In [ ]:
y=train_full['Etiqueta 2'].apply(categoricallabelAll)
y=y.to_numpy()
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6])

In [ ]:
from sklearn.model_selection import train_test_split

Xp_train,Xp_test,Xs_train,Xs_test,Xt_train,Xt_test,Xf_train,Xf_test,y_train,y_test=train_test_split(Xp,Xs,Xt,Xf,y,test_size=0.2,random_state=2023, stratify=y)

In [ ]:
Xp_train.shape,Xp_test.shape,Xs_train.shape,Xs_test.shape,Xt_train.shape,Xt_test.shape,Xf_train.shape,Xf_test.shape,y_train.shape,y_test.shape

((2767, 768),
 (692, 768),
 (2767, 768),
 (692, 768),
 (2767, 768),
 (692, 768),
 (2767, 768),
 (692, 768),
 (2767,),
 (692,))

# Keras model

In [ ]:
3//2

1

In [ ]:
from tensorflow.keras import backend as K
import gc
def ANNTC(input,base,pow_initial,num_max_blocks):

  drop_out=0.5
  print("base",base,"pow",pow_initial,"num_max_blocks",num_max_blocks)
  n=num_max_blocks//2
  neurons=int(base**(pow_initial+n+1))
  # Encoder
  x=None
  print("Encoder",n)
  try:
    for i in range(n):
      x=Dense(neurons)(input)
      x=BatchNormalization()(x)
      x=Activation('relu')(x)
      x=Dropout(drop_out)(x)
      input=x
      drop_out=0.2
      print("Block ",i,neurons)
      neurons=int(neurons/base)



    #BottleNeck
    x=Dense(neurons)(x)
    x=BatchNormalization()(x)
    x=Activation('relu')(x)
    x=Dropout(0.2)(x)
    print("BottleNeck",neurons)

    # Decoder
    print("Decoder",n)
    for i in range(n):
      neurons=int(neurons*base)
      print("Block ",i,neurons)
      x=Dense(neurons)(x)
      x=BatchNormalization()(x)
      x=Activation('relu')(x)
      x=Dropout(drop_out)(x)
      if i==n-1:
        drop_out=0.5
      drop_out=0.2
  except Exception as err:

    K.clear_session()
    gc.collect()
    del x
    print(f"Unexpected {err=}, {type(err)=}")
    raise

  return x


def neural_network(problem,start_input,trass_input,final_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_p=ANNTC(problem,base,pow_initial,num_max_blocks)
  mlp_s=ANNTC(start_input,base,pow_initial,num_max_blocks)
  mlp_t=ANNTC(trass_input,base,pow_initial,num_max_blocks)
  mlp_f=ANNTC(final_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_p, mlp_s, mlp_t,mlp_f])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[problem_input, start_input, trass_input,final_input], outputs=output)
  return model

In [ ]:
import numpy as np
np.linspace(5,70,14)


array([ 5., 10., 15., 20., 25., 30., 35., 40., 45., 50., 55., 60., 65.,
       70.])

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from time import time
#tensorflow random state
tf.random.set_seed(2023)

times=[]
max_accuracies=[]
max_val_accuracies=[]
max_val_losses=[]
max_losses=[]
learning_rates=[]


for patience in np.linspace(5,70,14):
  print("Paciencia",patience)
  problem_input=Input((768,))
  start_input=Input((768,))
  trass_input=Input((768,))
  final_input=Input((768,))


  my_model=neural_network(problem_input,start_input,trass_input,final_input,base=8,pow_initial=1, num_max_blocks=3)
  my_model.compile(loss='sparse_categorical_crossentropy',
                optimizer=Adam(),
                metrics=['accuracy'])

  es = EarlyStopping(monitor='val_loss', mode='min', patience=patience)

  mc = ModelCheckpoint('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/early_best_finall_patience{0}.keras'.format(patience), monitor='val_loss', mode='min', save_best_only=True)
  reduce_lr = ReduceLROnPlateau(
      monitor='val_loss',
      factor=0.1,
      patience=patience//2,
      min_lr=1e-9,
      verbose=1
  )

  time1=time()
  history=my_model.fit([Xp_train, Xs_train, Xt_train,Xf_train],y_train,batch_size=100,epochs=1000,
            validation_split=0.2,callbacks=[es, mc,reduce_lr])
  time2=time()

  df_histories=pd.DataFrame(history.history)
  df_histories.to_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/early_best_finall_{0}.csv'.format(patience))

  times.append(time2-time1)

  max_accuracies.append(max(history.history['accuracy']))
  max_val_accuracies.append(max(history.history['val_accuracy']))
  max_val_losses.append(max(history.history['val_loss']))
  max_losses.append(max(history.history['loss']))
  learning_rates.append(min(history.history['learning_rate']))

  print(f"{patience}")
  print("times=",time2-time1)
  print("max_val_accuracy",max(history.history['val_accuracy']))
  print("max_accuracy",max(history.history['accuracy']))


  print("max_val_loss",max(history.history['val_loss']))
  print("max_loss",max(history.history['loss']))
  print("learning_rate",min(history.history['learning_rate']))





Streaming output truncated to the last 5000 lines.
20/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9798 - loss: 0.2194
Epoch 102: ReduceLROnPlateau reducing learning rate to 1.0000001111620805e-07.
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9806 - loss: 0.2190 - val_accuracy: 0.9314 - val_loss: 0.3449 - learning_rate: 1.0000e-06
Epoch 103/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9802 - loss: 0.2155 - val_accuracy: 0.9314 - val_loss: 0.3452 - learning_rate: 1.0000e-07
Epoch 104/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9833 - loss: 0.2128 - val_accuracy: 0.9314 - val_loss: 0.3451 - learning_rate: 1.0000e-07
Epoch 105/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9853 - loss: 0.2105 - val_accuracy: 0.9314 - val_loss: 0.3453 - learning_rate: 1.0000e-07
Epoch 106/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9850 - loss: 0.2118 - val_accuracy: 0.9314 - val_loss: 0.3449 - learning_rate: 1.0000e-07
Epoch 107/1000
23/2

In [ ]:
df_histories=pd.DataFrame({"times":times,"max_accuracies":max_accuracies,"max_val_accuracies":max_val_accuracies,"max_val_losses":max_val_losses,"max_losses":max_losses,"learning_rates":learning_rates})
df_histories.to_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/early_best_finall.csv')

In [ ]:
df_histories

,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates
0,57.439719,0.957072,0.897112,2.320305,2.025782,1.000000e-09
1,69.617939,0.990511,0.940433,2.055581,2.025939,1.000000e-09
2,61.994090,0.994126,0.933213,2.037622,2.059239,1.000000e-08
3,64.311369,0.996837,0.938628,1.954160,2.023571,1.000000e-05
4,76.449078,0.998192,0.940433,1.929267,2.035504,1.000000e-06
5,70.008187,0.997741,0.938628,1.934476,2.009116,1.000000e-05
6,72.688126,0.998644,0.938628,1.860716,2.046122,1.000000e-06
7,90.695288,1.000000,0.944043,2.036217,2.014953,1.000000e-05
8,97.668076,0.999548,0.942238,1.916039,2.049077,1.000000e-06
9,80.442732,0.999548,0.936823,1.862118,1.998281,1.000000e-05


In [ ]:
from sklearn.metrics import matthews_corrcoef, precision_recall_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

def calculate_mcc_multiclass(y_true, y_pred_probs):
    y_pred_labels = np.argmax(y_pred_probs, axis=1)
    # Convertir etiquetas verdaderas one-hot a etiquetas enteras si es necesario
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    return matthews_corrcoef(y_true, y_pred_labels)

def calculate_auc_pr_multiclass(y_true, y_pred_probs, average='macro'):
    n_classes = y_pred_probs.shape[1]
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    y_true_bin = label_binarize(y_true, classes=range(n_classes))

    auc_pr_list = []
    for i in range(n_classes):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_probs[:, i])
        auc_pr_list.append(auc(recall, precision))

    if average == 'macro':
        return np.mean(auc_pr_list)
    elif average == 'weighted':
        class_counts = y_true_bin.sum(axis=0)
        return np.average(auc_pr_list, weights=class_counts)
    else:
        return auc_pr_list


In [ ]:
from tensorflow import keras
accuracies_predict=[]
loss_predict=[]
times_predict=[]
mccs_predict=[]
aucpr_predict=[]

for patience in np.linspace(5,70,14):
  model=keras.models.load_model('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/early_best_finall_patience{0}.keras'.format(patience))
  evaluate=model.evaluate([Xp_test,Xs_test,Xt_test,Xf_test],y_test)

  t1=time()
  y_pred=model.predict([Xp_test,Xs_test,Xt_test,Xf_test])
  t2=time()
  mcc=calculate_mcc_multiclass(y_test, y_pred)
  auc_pr=calculate_auc_pr_multiclass(y_test, y_pred)

  times_predict.append(t2-t1)
  accuracies_predict.append(evaluate[1])
  loss_predict.append(evaluate[0])
  mccs_predict.append(mcc)
  aucpr_predict.append(auc_pr)


  accuracy=evaluate[1]
  loss=evaluate[0]

  print("accuracy",accuracy)
  print("loss",loss)
  print("mcc",mcc)
  print("auc_pr",auc_pr)
  print("time predict",t2-t1)

22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.8881 - loss: 0.5322
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step
accuracy 0.8858381509780884
loss 0.5423533320426941
mcc 0.8473724602454143
auc_pr 0.853530403434283
time predict 1.6022374629974365


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.9160 - loss: 0.3338
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step
accuracy 0.913294792175293
loss 0.35766708850860596
mcc 0.8841744384780943
auc_pr 0.8806096637963878
time predict 1.6032922267913818


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.9277 - loss: 0.2813
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step
accuracy 0.9176300764083862
loss 0.3089827001094818
mcc 0.8899282951302022
auc_pr 0.8854002873371951
time predict 1.5824129581451416


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.9215 - loss: 0.2578
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step
accuracy 0.9190751314163208
loss 0.31486693024635315
mcc 0.8922526893443669
auc_pr 0.8917123080599559
time predict 1.5418164730072021


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.9212 - loss: 0.2316
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step
accuracy 0.9248554706573486
loss 0.2711978256702423
mcc 0.8997230156453611
auc_pr 0.8904548380271918
time predict 1.5248589515686035


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


In [ ]:
df_histories['predict_accuracies']=accuracies_predict
df_histories['predict_losses']=loss_predict
df_histories['times_predict']=times_predict
df_histories['mccs']=mccs_predict
df_histories['aucpr']=aucpr_predict
df_histories.to_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/early_best_finall_predict.csv')

In [ ]:
df_histories["patience"]=np.linspace(5,70,14)

In [ ]:
df_histories

,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,predict_accuracies,predict_losses,times_predict,mccs,aucpr,patience
0,57.439719,0.957072,0.897112,2.320305,2.025782,1.000000e-09,0.885838,0.542353,1.602237,0.847372,0.853530,5.0
1,69.617939,0.990511,0.940433,2.055581,2.025939,1.000000e-09,0.913295,0.357667,1.603292,0.884174,0.880610,10.0
2,61.994090,0.994126,0.933213,2.037622,2.059239,1.000000e-08,0.917630,0.308983,1.582413,0.889928,0.885400,15.0
3,64.311369,0.996837,0.938628,1.954160,2.023571,1.000000e-05,0.921965,0.289377,1.549080,0.895884,0.890299,20.0
4,76.449078,0.998192,0.940433,1.929267,2.035504,1.000000e-06,0.916185,0.287139,1.572104,0.888122,0.886586,25.0
5,70.008187,0.997741,0.938628,1.934476,2.009116,1.000000e-05,0.924855,0.288177,1.538031,0.899703,0.889111,30.0
6,72.688126,0.998644,0.938628,1.860716,2.046122,1.000000e-06,0.914740,0.296201,1.515040,0.886222,0.887111,35.0
7,90.695288,1.000000,0.944043,2.036217,2.014953,1.000000e-05,0.923410,0.281967,1.564830,0.897825,0.890658,40.0
8,97.668076,0.999548,0.942238,1.916039,2.049077,1.000000e-06,0.924855,0.278860,1.613987,0.899778,0.893428,45.0
9,80.442732,0.999548,0.936823,1.862118,1.998281,1.000000e-05,0.926301,0.288960,1.584160,0.901734,0.885882,50.0


In [ ]:
df_histories[(df_histories['aucpr']>0.889)]

,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,predict_accuracies,predict_losses,times_predict,mccs,aucpr,patience
3,64.311369,0.996837,0.938628,1.954160,2.023571,0.000010,0.921965,0.289377,1.549080,0.895884,0.890299,20.0
5,70.008187,0.997741,0.938628,1.934476,2.009116,0.000010,0.924855,0.288177,1.538031,0.899703,0.889111,30.0
7,90.695288,1.000000,0.944043,2.036217,2.014953,0.000010,0.923410,0.281967,1.564830,0.897825,0.890658,40.0
8,97.668076,0.999548,0.942238,1.916039,2.049077,0.000001,0.924855,0.278860,1.613987,0.899778,0.893428,45.0
10,88.870099,1.000000,0.940433,1.942262,1.999402,0.000001,0.921965,0.283506,1.540994,0.895746,0.889937,55.0
11,93.134187,1.000000,0.945848,2.082152,2.061558,0.000010,0.919075,0.280132,1.569605,0.891997,0.893560,60.0
12,97.958311,1.000000,0.945848,1.926063,1.965274,0.000010,0.919075,0.314867,1.541816,0.892253,0.891712,65.0
13,89.605544,0.999548,0.938628,1.991907,2.028588,0.000010,0.924855,0.271198,1.524859,0.899723,0.890455,70.0


In [ ]:
df_histories[(df_histories['aucpr']>0.889)&(df_histories['max_accuracies']-df_histories['max_val_accuracies']<0.055)]

,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,predict_accuracies,predict_losses,times_predict,mccs,aucpr,patience
11,93.134187,1.0,0.945848,2.082152,2.061558,0.00001,0.919075,0.280132,1.569605,0.891997,0.893560,60.0
12,97.958311,1.0,0.945848,1.926063,1.965274,0.00001,0.919075,0.314867,1.541816,0.892253,0.891712,65.0


Early stoping 40 the best